# 05 — Prebuilt ID Document: KYC Identity Verification

**Time**: ~5 minutes · **Model**: `prebuilt-idDocument` · **No training required**

---

## Insurance Scenario

During **onboarding** or **claim filing**, policyholders submit identity documents — driver's licenses, passports, or national IDs. Your system needs to:

- Verify the policyholder's identity (KYC compliance)
- Extract name, date of birth, and ID number
- Check document expiration dates
- Match against policy records

---

## Prerequisites

Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb) first.

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult
import pandas as pd

load_dotenv(dotenv_path=os.path.join("..", ".env"))

_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
_credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"],
    credential=_credential
)
print("✅ Client ready")

## Analyze an ID Document

In [ ]:
# Sample driver's license
id_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/DriverLicense.png"

poller = client.begin_analyze_document(
    "prebuilt-idDocument",
    AnalyzeDocumentRequest(url_source=id_url)
)
result: AnalyzeResult = poller.result()

print(f"ID documents found: {len(result.documents) if result.documents else 0}")

## Extract Identity Fields

In [ ]:
if result.documents:
    id_doc = result.documents[0]
    f = id_doc.fields
    
    print("=" * 50)
    print("IDENTITY DOCUMENT")
    print("=" * 50)
    print(f"  Document type:    {id_doc.doc_type}")
    print(f"  Confidence:       {id_doc.confidence:.2%}")
    print()
    
    # Personal information
    key_fields = [
        ("FirstName", "First Name"),
        ("LastName", "Last Name"),
        ("DateOfBirth", "Date of Birth"),
        ("Sex", "Sex"),
        ("Address", "Address"),
        ("DocumentNumber", "Document Number"),
        ("DateOfExpiration", "Expiration Date"),
        ("DateOfIssue", "Issue Date"),
        ("CountryRegion", "Country/Region"),
        ("Region", "State/Region"),
    ]
    
    for field_key, display_name in key_fields:
        field = f.get(field_key, {})
        value = field.get("content", "Not found")
        confidence = field.get("confidence", 0)
        flag = "⚠️" if 0 < confidence < 0.80 else "✅" if confidence >= 0.80 else "  "
        print(f"  {flag} {display_name:20s} {str(value):30s} ({confidence:.0%})")

## KYC Validation Checks

**Insurance-specific**: Automatically run common KYC validation rules.

In [ ]:
from datetime import datetime

if result.documents:
    id_doc = result.documents[0]
    f = id_doc.fields
    
    print("KYC VALIDATION CHECKS")
    print("=" * 50)
    
    # Check 1: Document confidence
    conf = id_doc.confidence
    if conf and conf >= 0.80:
        print(f"  ✅ Document confidence: {conf:.2%} (acceptable)")
    else:
        print(f"  ❌ Document confidence: {conf:.2%} (needs manual review)")
    
    # Check 2: Name fields present
    first = f.get("FirstName", {}).get("content")
    last = f.get("LastName", {}).get("content")
    if first and last:
        print(f"  ✅ Name extracted: {first} {last}")
    else:
        print(f"  ❌ Name missing — manual verification required")
    
    # Check 3: Document number present
    doc_num = f.get("DocumentNumber", {}).get("content")
    if doc_num:
        print(f"  ✅ Document number: {doc_num}")
    else:
        print(f"  ❌ Document number not found")
    
    # Check 4: Expiration date
    exp = f.get("DateOfExpiration", {}).get("content")
    if exp:
        print(f"  ℹ️  Expiration: {exp} (verify manually that document is not expired)")
    else:
        print(f"  ⚠️ Expiration date not found")
    
    print()
    print("Note: Always verify ID documents against original sources in production.")

---

## Key Takeaways

| Feature | Value for Insurance |
|---|---|
| **Name extraction** | Match against policy records for KYC |
| **DOB extraction** | Verify age for age-dependent policies |
| **Document number** | Cross-reference with government databases |
| **Expiration check** | Reject expired IDs during onboarding |

## Supported ID Types

- US driver's licenses
- International passports
- National identity cards (many countries)
- US state IDs

Full list: [ID Document model documentation](https://learn.microsoft.com/en-us/azure/ai-services/document-intelligence/prebuilt/id-document?view=doc-intel-4.0.0)

## Next Steps

| Notebook | What You'll Learn |
|---|---|
| [06-prebuilt-insurance-card.ipynb](06-prebuilt-insurance-card.ipynb) | Extract health insurance card details |
| [10-human-in-the-loop.ipynb](10-human-in-the-loop.ipynb) | Build a confidence-based review workflow |